<h1>Space X  Falcon 9 First Stage Landing Prediction</h2>


<h2>Exploratory Data Analysis With SQL</h2>

In [ ]:
!pip install sqlalchemy==1.3.9  #Used to connect Python with databases


  Using cached SQLAlchemy-1.3.9.tar.gz (6.0 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for sqlalchemy: filename=sqlalchemy-1.3.9-cp313-cp313-win_amd64.whl size=1143040 sha256=4a800dbcde9e5f32a0da75419863967d8dcb5510bc6deb59d669214a6db053ee
  Stored in directory: c:\users\dell\appdata\local\pip\cache\wheels\7f\0c\92\6ec5ca28eb22ed22341110143de753ec622045fbfc750edb26
Successfully built sqlalchemy
  Attempting uninstall: sqlalchemy
    Found existing installation: SQLAlchemy 1.3.9
    Uninstalling SQLAlchemy-1.3.9:
      Successfully uninstalled SQLAlchemy-1.3.9


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aext-project-filebrowser-server 4.20.0 requires sqlalchemy>=2.0.29, but you have sqlalchemy 1.3.9 which is incompatible.
alembic 1.17.2 requires SQLAlchemy>=1.4.0, but you have sqlalchemy 1.3.9 which is incompatible.


In [ ]:
!pip install ipython-sql   #ipython: Lets you write SQL queries directly inside Jupyter Notebook
!pip install ipython-sql prettytable   #prettytable: Formats query results into nice tables

In [ ]:
%load_ext sql  #loads sql extension

In [4]:
%sql sqlite:///my_data1.db

In [ ]:
import csv, sqlite3
import prettytable
prettytable.DEFAULT = 'DEFAULT'
#Database connection
con = sqlite3.connect("my_data1.db")
cur = con.cursor()

In [7]:
!pip install -q pandas

In [8]:
import pandas as pd
df=pd.read_csv("Spacex.csv");
df.to_sql("SPACEXTBL", con, if_exists='replace', index=False,method="multi")

101

In [9]:
#DROP THE TABLE IF EXISTS
%sql DROP TABLE IF EXISTS SPACEXTABLE;

 * sqlite:///my_data1.db
Done.


[]

In [10]:
%sql create table SPACEXTABLE as select * from SPACEXTBL where Date is not null

 * sqlite:///my_data1.db
Done.


[]

Display the names of the unique launch sites  in the space mission

In [11]:
%sql Select Launch_Site from SPACEXTBL 

 * sqlite:///my_data1.db
Done.


Launch_Site
CCAFS LC-40
CCAFS LC-40
CCAFS LC-40
CCAFS LC-40
CCAFS LC-40
VAFB SLC-4E
CCAFS LC-40
CCAFS LC-40
CCAFS LC-40
CCAFS LC-40


Display 5 records where launch sites begin with the string 'CCA' 

In [12]:
%sql SELECT * FROM SPACEXTBL where Launch_Site LIKE 'CCA%' LIMIT 5

 * sqlite:///my_data1.db
Done.


Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of Brouere cheese",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


Display the total payload mass carried by boosters launched by NASA (CRS)

In [22]:
%sql SELECT SUM(PAYLOAD_MASS__KG_) AS 'Sum of Payload Mass' FROM SPACEXTBL WHERE Customer='NASA (CRS)'

 * sqlite:///my_data1.db
Done.


Sum of Payload Mass
45596


 Display average payload mass carried by booster version F9 v1.1

In [21]:
%sql SELECT AVG(PAYLOAD_MASS__KG_) AS 'Average of Payload_mass' FROM SPACEXTBL WHERE Booster_Version='F9 v1.1'

 * sqlite:///my_data1.db
Done.


Average of Payload_mass
2928.4


List the date when the first succesful landing outcome in ground pad was acheived.

In [23]:
%sql SELECT min(Date) AS 'First Successful Landing' FROM SPACEXTBL WHERE Landing_Outcome='Success (ground pad)'

 * sqlite:///my_data1.db
Done.


First Successful Landing
2015-12-22


List the names of the boosters which have success in drone ship and have payload mass greater than 4000 but less than 6000

In [24]:
%sql SELECT Booster_Version FROM SPACEXTBL WHERE Landing_Outcome='Success (drone ship)' AND PAYLOAD_MASS__KG_>4000 AND PAYLOAD_MASS__KG_<6000

 * sqlite:///my_data1.db
Done.


Booster_Version
F9 FT B1022
F9 FT B1026
F9 FT B1021.2
F9 FT B1031.2


List the total number of successful and failure mission outcomes

In [28]:
%sql SELECT Mission_Outcome,COUNT(*) FROM SPACEXTBL GROUP BY Mission_Outcome

 * sqlite:///my_data1.db
Done.


Mission_Outcome,COUNT(*)
Failure (in flight),1
Success,98
Success,1
Success (payload status unclear),1


List all the booster_versions that have carried the maximum payload mass, using a subquery with a suitable aggregate function.

In [30]:
%sql SELECT Booster_Version FROM SPACEXTBL WHERE PAYLOAD_MASS__KG_=(SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL)

 * sqlite:///my_data1.db
Done.


Booster_Version
F9 B5 B1048.4
F9 B5 B1049.4
F9 B5 B1051.3
F9 B5 B1056.4
F9 B5 B1048.5
F9 B5 B1051.4
F9 B5 B1049.5
F9 B5 B1060.2
F9 B5 B1058.3
F9 B5 B1051.6


List the records which will display the month names, failure landing_outcomes in drone ship ,booster versions, launch_site for the months in year 2015.

In [34]:
%sql SELECT substr(Date,6,2) AS Month,Landing_Outcome,Booster_Version,Launch_Site FROM SPACEXTBL WHERE substr(Date,1,4) = '2015'

 * sqlite:///my_data1.db
Done.


Month,Landing_Outcome,Booster_Version,Launch_Site
01,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
02,Controlled (ocean),F9 v1.1 B1013,CCAFS LC-40
03,No attempt,F9 v1.1 B1014,CCAFS LC-40
04,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40
04,No attempt,F9 v1.1 B1016,CCAFS LC-40
06,Precluded (drone ship),F9 v1.1 B1018,CCAFS LC-40
12,Success (ground pad),F9 FT B1019,CCAFS LC-40


Rank the count of landing outcomes (such as Failure (drone ship) or Success (ground pad)) between the date 2010-06-04 and 2017-03-20, in descending order.

In [37]:
%sql SELECT Landing_Outcome, COUNT(*) AS COUNT FROM SPACEXTBL WHERE DATE BETWEEN('2010-06-04') AND '2017-03-20' GROUP BY Landing_Outcome ORDER BY count DESC

 * sqlite:///my_data1.db
Done.


Landing_Outcome,COUNT
No attempt,10
Success (drone ship),5
Failure (drone ship),5
Success (ground pad),3
Controlled (ocean),3
Uncontrolled (ocean),2
Failure (parachute),2
Precluded (drone ship),1
